In [ ]:
import json
import os
import random
import re
from ast import literal_eval
from pathlib import Path

import flair
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from flair.data import Corpus, Sentence
from flair.datasets import ColumnCorpus
from flair.embeddings import (
    FlairEmbeddings,
    StackedEmbeddings,
    TransformerWordEmbeddings,
)
from flair.models import SequenceTagger
from flair.tokenization import SpaceTokenizer
from flair.trainers import ModelTrainer
from flair.visual.training_curves import Plotter
from rich.pretty import pprint
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import OneCycleLR

import langextract as lx
from aymurai.database.utils import text_to_uuid
from langextract.providers.openai import OpenAILanguageModel

# --- Configuration ---
flair.device = torch.device("cuda")
flair.device = torch.device("mps") 

print(f"CUDA Available: {torch.cuda.is_available()}")

## Dataset

In [ ]:
# define columns
columns = {0: "text", 1: "ner"}

# this is the folder in which train, test and dev files reside
data_folder = "/Users/MacConra/Documents/Collective/AymurAI/backend/resources/data/restricted/ner-review/labeled-dataset/"

# 1. init a corpus using column format, data folder and the names of the train, dev and test files
corpus = ColumnCorpus(
    data_folder,
    columns,
    train_file="train.txt",
    test_file="test.txt",
    dev_file="dev.txt",
)

In [ ]:
# for i in range(10):
#     print(corpus.train[i])

In [ ]:
# for i in range(10):
#     print(corpus.dev[i])

In [ ]:
# for i in range(10):
#     print(corpus.test[i])

In [ ]:
# 2. what label do we want to predict?
label_type = "ner"

In [ ]:
# 3. make the label dictionary from the corpus
vocab_dictionary = corpus.make_vocab_dictionary()
print(vocab_dictionary)

In [ ]:
# 4. make the vocab dictionary from the corpus
label_dictionary = corpus.make_label_dictionary(label_type=label_type, add_unk=True)

In [ ]:
print(corpus.obtain_statistics())

In [ ]:
stats = literal_eval(corpus.obtain_statistics())

In [ ]:
pd.Series(stats["TRAIN"]["number_of_documents_per_class"]).sort_values(
    ascending=False
).plot(kind="bar", title="Train set - number of documents per label")

In [ ]:
len(stats["TRAIN"]["number_of_documents_per_class"].keys())

In [ ]:
pd.Series(stats["DEV"]["number_of_documents_per_class"]).sort_values(
    ascending=False
).plot(kind="bar", title="Dev set - number of documents per label")

In [ ]:
len(stats["DEV"]["number_of_documents_per_class"].keys())

In [ ]:
set(stats["TRAIN"]["number_of_documents_per_class"].keys()).symmetric_difference(
    set(stats["DEV"]["number_of_documents_per_class"].keys())
)

In [ ]:
pd.Series(stats["TEST"]["number_of_documents_per_class"]).sort_values(
    ascending=False
).plot(kind="bar", title="Test set - number of documents per label")

In [ ]:
len(stats["TEST"]["number_of_documents_per_class"].keys())

In [ ]:
set(stats["TRAIN"]["number_of_documents_per_class"].keys()).symmetric_difference(
    set(stats["TEST"]["number_of_documents_per_class"].keys())
)

## Evaluation

In [ ]:
# path = "/resources/ner/flair/anonymizer"

In [ ]:
# # load model
# tagger = SequenceTagger.load(f"{path}/best-model.pt")

In [ ]:
# # rewrite `label_dictionary` attribute to handle unknown items
# tagger.label_dictionary = label_dictionary

In [ ]:
# evaluation = tagger.evaluate(
#     corpus.train,
#     label_type,
#     path + "/train-evaluation.txt",
# )

In [ ]:
# print(evaluation.main_score, evaluation.loss)

In [ ]:
# print(evaluation.detailed_results)

In [ ]:
# evaluation = tagger.evaluate(
#     corpus.dev,
#     label_type,
#     path + "/dev-evaluation.txt",
# )

In [ ]:
# print(evaluation.main_score, evaluation.loss)

In [ ]:
# print(evaluation.detailed_results)

In [ ]:
# evaluation = tagger.evaluate(
#     corpus.test,
#     label_type,
#     path + "/test-evaluation.txt",
# )

In [ ]:
# print(evaluation.main_score, evaluation.loss)

In [ ]:
# print(evaluation.detailed_results)

In [ ]:
pd.set_option("display.max_rows", 100)

path = "ner-evaluations"
df_test = pd.read_csv(f"{path}/test-evaluation.txt", sep="\s", header=None)
df_test.columns = ["token", "label", "pred"]
df_test.head()

# df_train_ner = pd.read_csv(f"{path}/train-evaluation.txt", sep="\s", header=None)
# df_train_ner.columns = ["token", "label", "ner_pred"]
# df_train_ner.head()

In [ ]:
df_test.info()

In [ ]:
df_test["label"].value_counts(normalize=True)

In [ ]:
# Exact match
df_test["match"] = df_test["label"] == df_test["pred"]
df_test["match"].value_counts(normalize=True)

In [ ]:
df_test.loc[df_test["label"] != "O", "match"].value_counts(normalize=True)

In [ ]:
df_test.loc[df_test["label"] == "O", "match"].value_counts(normalize=True)

In [ ]:
df_test.loc[(df_test["label"] == "O") & (df_test["match"] != 1)]

In [ ]:
normalize_class = lambda x: re.sub(r"B-|I-", "", x)

In [ ]:

df_test["normalized_label"] = df_test["label"].map(normalize_class)
df_test["normalized_pred"] = df_test["pred"].map(normalize_class)

In [ ]:
df_test.head()

In [ ]:
# Normalized exact match
df_test["normalized_match"] = df_test["normalized_label"] == df_test["normalized_pred"]
df_test["normalized_match"].value_counts(normalize=True)

In [ ]:
df_test.loc[df_test["normalized_label"] != "O", "normalized_match"].value_counts(normalize=True)

In [ ]:
df_test.loc[df_test["normalized_label"] == "O", "normalized_match"].value_counts(normalize=True)

In [ ]:
df_test["normalized_pred"].value_counts(normalize=True)

In [ ]:
print(classification_report(df_test["label"], df_test["pred"]))

In [ ]:
print(classification_report(df_test["normalized_label"], df_test["normalized_pred"]))

In [ ]:
plt.figure(figsize=(20, 20))

labels = df_test["normalized_label"].unique()

cm = confusion_matrix(
    df_test["normalized_label"],
    df_test["normalized_pred"],
    labels=labels,
    normalize="true",
)

sns.heatmap(
    cm,
    vmin=0.0,
    vmax=1.0,
    cmap="Blues",
    annot=True,
    fmt=".2f",
    cbar=False,
    xticklabels=labels,
    yticklabels=labels,
)

plt.title("Confusion Matrix", fontdict={"fontsize": 20})

## Langextract evaluation

In [ ]:
with open(f"labeled-dataset/test.txt", "r") as f:
    test_text = f.readlines()

# print(test_text[:100])

In [ ]:
with open(f"labeled-dataset/train.txt", "r") as f:
    train_text = f.readlines()

In [ ]:
# Function to parse paragraphs from token-label lines
def parse_paragraphs(lines):
    paragraphs = []
    current_tokens = []
    current_labels = []

    for raw_line in lines:
        line = raw_line.rstrip()
        if not line:
            if current_tokens:
                paragraph_text = " ".join(current_tokens).strip()
                paragraphs.append((paragraph_text, list(current_labels)))
                current_tokens = []
                current_labels = []
            continue

        try:
            token, label = line.rsplit(" ", 1)
        except ValueError:
            token = line
            label = ""

        current_tokens.append(token)
        current_labels.append(label.strip())

    if current_tokens:
        paragraph_text = " ".join(current_tokens).strip()
        paragraphs.append((paragraph_text, list(current_labels)))

    return paragraphs

In [ ]:
def return_paragraphs(lines):
    paragraphs = []
    current_tokens = []

    for raw_line in lines:
        line = raw_line.rstrip()
        if not line:
            if current_tokens:
                paragraph_text = " ".join(current_tokens).strip()
                paragraphs.append((paragraph_text))
                current_tokens = []
            continue

        try:
            token, label = line.rsplit(" ", 1)
        except ValueError:
            token = line

        current_tokens.append(token)

    if current_tokens:
        paragraph_text = " ".join(current_tokens).strip()
        paragraphs.append((paragraph_text))

    return paragraphs

In [ ]:
test_paragraphs = return_paragraphs(test_text)
print(len(test_paragraphs))
train_paragraphs = return_paragraphs(train_text)
print(len(train_paragraphs))

In [ ]:
test = parse_paragraphs(test_text)
print(len(test))

train = parse_paragraphs(train_text)
print(len(train))

In [ ]:
# output_dir = Path("langextract-outputs")
# outputs = [file for file in list(output_dir.iterdir()) if file.suffix == ".jsonl"]
# len(outputs)

In [ ]:
# langextract_paragraphs = []
# for output in outputs:
#     document_id = output.stem

#     extracted_data = next(
#         lx.io.load_annotated_documents_jsonl(output_dir / f"{document_id}.jsonl"), None
#     )
#     langextract_paragraphs.append(extracted_data.text)

In [ ]:
# len(langextract_paragraphs)

In [ ]:
# left_to_langextract_inference = list(set(test_paragraphs) - set(langextract_paragraphs))
# len(left_to_langextract_inference)

In [ ]:
# set(left_to_langextract_inference)

In [ ]:
# m_left_to_langextract_inference = list(set(langextract_paragraphs)-set(test_paragraphs))
# len(m_left_to_langextract_inference)

In [ ]:
# for item in left_to_langextract_inference:
#     for idx, paragraph in enumerate(langextract_paragraphs):
#         if item[10:-1] in paragraph:
#             print(idx)
#             print(paragraph)

In [ ]:
# prompt = """
#     Sos un asistente especializado en el análisis de documentos judiciales.
#     Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
#     Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

#     - "BANCO": Nombre de una entidad bancaria, pública o privada.
#     - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
#     - "CORREO_ELECTRONICO": Dirección de correo electrónico.
#     - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
#     - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
#     - "DIRECCION": Dirección postal específica (calle, número, etc.).
#     - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
#     - "EDAD": Edad explícita de una persona.
#     - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
#     - "FECHA": Fecha completa o parcial (día, mes y/o año).
#     - "LINK": Enlace o URL a una página web.
#     - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
#     - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
#     - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
#     - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
#     - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
#     - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
#     - "NUM_MATRICULA": Número de matrícula profesional o académica.
#     - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
#     - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
#     - "TELEFONO": Número telefónico (fijo o celular).
# """

In [ ]:
# # 2. Provide a high-quality example to guide the model
# examples = [
#     lx.data.ExampleData(
#         text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="FECHA", extraction_text="5 de mayo de 2023"
#             ),
#             lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
#             lx.data.Extraction(
#                 extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
#             ),
#             lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
#         extractions=[
#             lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Gómez"),
#             lx.data.Extraction(
#                 extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
#             ),
#             lx.data.Extraction(
#                 extraction_class="CUIJ", extraction_text="12-34567890-1"
#             ),
#             lx.data.Extraction(
#                 extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
#         extractions=[
#             lx.data.Extraction(extraction_class="PER", extraction_text="Miguel Torres"),
#             lx.data.Extraction(extraction_class="DNI", extraction_text="30123456"),
#             lx.data.Extraction(extraction_class="FECHA", extraction_text="14/02/1990"),
#             lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
#             lx.data.Extraction(
#                 extraction_class="NACIONALIDAD", extraction_text="paraguaya"
#             ),
#             lx.data.Extraction(
#                 extraction_class="ESTUDIOS",
#                 extraction_text="estudios secundarios completos",
#             ),
#             lx.data.Extraction(
#                 extraction_class="DIRECCION",
#                 extraction_text="Av. Corrientes 3456",
#             ),
#             lx.data.Extraction(extraction_class="PER", extraction_text="Jorge Pérez"),
#             lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
#             lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
#             lx.data.Extraction(
#                 extraction_class="PER", extraction_text="Laura Fernández"
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
#         extractions=[
#             lx.data.Extraction(extraction_class="PER", extraction_text="Juan López"),
#             lx.data.Extraction(
#                 extraction_class="TELEFONO", extraction_text="11-2345-6789"
#             ),
#             lx.data.Extraction(extraction_class="PER", extraction_text="Ana García"),
#             lx.data.Extraction(
#                 extraction_class="NUM_MATRICULA", extraction_text="12345"
#             ),
#             lx.data.Extraction(
#                 extraction_class="MARCA_AUTOMOVIL",
#                 extraction_text="Volkswagen",
#             ),
#             lx.data.Extraction(
#                 extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
#             ),
#             lx.data.Extraction(
#                 extraction_class="CBU",
#                 extraction_text="2850590940090412345671",
#             ),
#             lx.data.Extraction(
#                 extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
#             ),
#             lx.data.Extraction(
#                 extraction_class="BANCO", extraction_text="Banco Nación"
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="CORREO_ELECTRONICO",
#                 extraction_text="fiscalia.central@justicia.gob.ar",
#             ),
#             lx.data.Extraction(
#                 extraction_class="LINK",
#                 extraction_text="https://justicia.gob.ar/actuaciones",
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle Sarmiento 1234, localidad de Moreno.",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="FECHA", extraction_text="5 de mayo de 2023"
#             ),
#             lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
#             lx.data.Extraction(
#                 extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
#             ),
#             lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="MARCA_AUTOMOVIL",
#                 extraction_text="Volkswagen Voyage",
#             ),
#             lx.data.Extraction(
#                 extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"
#             ),
#         ],
#     ),
#     lx.data.ExampleData(
#         text="A su vez, requirió informes al Banco BBVA Francés, respecto de las cuentas bancarias de la denunciante, Carla Alejandra Garcia, D.N.I. 36.998.621 identificadas como Caja de ahorro en pesos argentinos número 117-59824/6 con CBU 0180132640000004685591 y Caja de ahorro en dólares número 119-619018/2 con CBU 0170115544000062081822.",
#         extractions=[
#             lx.data.Extraction(
#                 extraction_class="PER", extraction_text="Carla Alejandra Garcia"
#             ),
#             lx.data.Extraction(extraction_class="DNI", extraction_text="36.998.621"),
#             lx.data.Extraction(
#                 extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"
#             ),
#             lx.data.Extraction(
#                 extraction_class="CBU", extraction_text="0180132640000004685591"
#             ),
#             lx.data.Extraction(
#                 extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"
#             ),
#             lx.data.Extraction(
#                 extraction_class="CBU", extraction_text="0170115544000062081822"
#             ),
#         ],
#     ),
# ]

In [ ]:
model_qwen= OpenAILanguageModel(
    model_id="qwen3:8b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

In [ ]:
# for text in left_to_langextract_inference[2:3]:
#     try:
#         result = lx.extract(
#             text_or_documents=text,
#             prompt_description=prompt,
#             examples=examples,
#             model=model_qwen,
#             use_schema_constraints=False
#         )
#         pprint(result)
#     except Exception as e:
#         print(f"Error de formato: {e}")

In [ ]:
# Custom OpenAI model class to handle Ollama gpt-oss:20b
class CustomOpenAIModel(OpenAILanguageModel):
    def _process_single_prompt(self, prompt, config):
        api_params = {
            "model": self.model_id,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 2000,
        }

        try:
            response = self._client.chat.completions.create(**api_params)
            output_text = response.choices[0].message.content
            return lx.inference.ScoredOutput(score=1.0, output=output_text)

        except Exception as e:
            raise lx.exceptions.InferenceRuntimeError(
                f"Custom OpenAI API error: {str(e)}", original=e
            ) from e


# Instantiate the custom model
model_gpt = CustomOpenAIModel(
    model_id="gpt-oss:20b",
    api_key="dummy",
    base_url="http://localhost:11434/v1",
)

# # Run extraction with the custom model
# result = lx.extract(
#     text_or_documents=text,
#     prompt_description=prompt,
#     examples=examples,
#     model=model_gpt,
#     fence_output=False,
#     use_schema_constraints=False,
#     fetch_urls=False,
# )
# pprint(result)

In [ ]:
# # --- MLflow Configuration ---
# MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5005")
# EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT", "left-to-inference-gpt-vs-qwen")

# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# mlflow.set_experiment(EXPERIMENT_NAME)

# # --- Output Directories ---
# output_dir_gpt = Path("langextract-outputs-left-gpt")
# output_dir_gpt.mkdir(exist_ok=True)

# output_dir_qwen = Path("langextract-outputs-left-qwen")
# output_dir_qwen.mkdir(exist_ok=True)

# # --- Main Inference Loop with Tracing ---
# with mlflow.start_run(run_name="GPT_vs_Qwen_Inference_Run"):
#     # Log global parameters for the run
#     mlflow.log_param("gpt_model", str(model_gpt))
#     mlflow.log_param("qwen_model", str(model_qwen))
#     mlflow.log_param("total_docs_to_process", len(left_to_langextract_inference))

#     for text in left_to_langextract_inference:
#         file_id = text_to_uuid(text).hex
        
#         # Start a Parent Span for this specific document in the Traces tab
#         with mlflow.start_span(name=f"Doc_{file_id[:8]}") as doc_span:
#             doc_span.set_attribute("input_text_snippet", text[:150])
            
#             try:
#                 # --- PHASE 1: GPT INFERENCE ---
#                 with mlflow.start_span(name="gpt_extract") as gpt_span:
#                     result_gpt = lx.extract(
#                         text_or_documents=text,
#                         prompt_description=prompt,
#                         examples=examples,
#                         model=model_gpt,
#                         use_schema_constraints=False
#                     )
#                     gpt_span.set_attribute("status", "success")
#                     gpt_span.set_attribute("extractions_found", len(result_gpt.extractions))

#                 # Prepare JSONL structure
#                 output_data = {
#                     "extractions": [
#                         {
#                             "extraction_class": e.extraction_class,
#                             "extraction_text": e.extraction_text,
#                             "char_interval": {
#                                 "start_pos": e.char_interval.start_pos if e.char_interval else None,
#                                 "end_pos": e.char_interval.end_pos if e.char_interval else None
#                             },
#                             "alignment_status": e.alignment_status.value if hasattr(e, 'alignment_status') else None,
#                             "extraction_index": e.extraction_index,
#                             "group_index": e.group_index,
#                             "description": e.description,
#                             "attributes": e.attributes if hasattr(e, 'attributes') else {}
#                         } for e in result_gpt.extractions
#                     ],
#                     "text": text,
#                     "document_id": file_id
#                 }
                
#                 file_path_gpt = output_dir_gpt / f"{file_id}.jsonl"
#                 with open(file_path_gpt, "w", encoding="utf-8") as f:
#                     f.write(json.dumps(output_data, ensure_ascii=False) + "\n")
                
#                 mlflow.log_metric("gpt_success", 1)
#                 print(f"GPT Success: {file_path_gpt.name}")

#             except Exception as e_gpt:
#                 # Log GPT failure to the Trace
#                 doc_span.set_attribute("gpt_error", str(e_gpt))
#                 mlflow.log_metric("gpt_error", 1)
#                 print(f"GPT Failed ({file_id[:8]}...): {e_gpt}. Trying Qwen...")

#                 try:
#                     # --- PHASE 2: QWEN FALLBACK ---
#                     with mlflow.start_span(name="qwen_fallback_extract") as qwen_span:
#                         result_qwen = lx.extract(
#                             text_or_documents=text,
#                             prompt_description=prompt,
#                             examples=examples,
#                             model=model_qwen,
#                             use_schema_constraints=False
#                         )
#                         qwen_span.set_attribute("status", "success")
#                         qwen_span.set_attribute("extractions_found", len(result_qwen.extractions))

#                     output_data_qwen = {
#                         "extractions": [
#                             {
#                                 "extraction_class": e.extraction_class,
#                                 "extraction_text": e.extraction_text,
#                                 "char_interval": {
#                                     "start_pos": e.char_interval.start_pos if e.char_interval else None,
#                                     "end_pos": e.char_interval.end_pos if e.char_interval else None
#                                 },
#                                 "alignment_status": e.alignment_status.value if hasattr(e, 'alignment_status') else None,
#                                 "extraction_index": e.extraction_index,
#                                 "group_index": e.group_index,
#                                 "description": e.description,
#                                 "attributes": e.attributes if hasattr(e, 'attributes') else {}
#                             } for e in result_qwen.extractions
#                         ],
#                         "text": text,
#                         "document_id": file_id
#                     }
                    
#                     file_path_qwen = output_dir_qwen / f"{file_id}.jsonl"
#                     with open(file_path_qwen, "w", encoding="utf-8") as f:
#                         f.write(json.dumps(output_data_qwen, ensure_ascii=False) + "\n")
                    
#                     mlflow.log_metric("qwen_success", 1)
#                     print(f"Qwen Fallback Saved: {file_path_qwen.name}")

#                 except Exception as e_qwen:
#                     # Log total failure to the Trace
#                     doc_span.set_attribute("qwen_error", str(e_qwen))
#                     mlflow.log_metric("total_failure", 1)
#                     print(f"Critical Error: Both models failed for {file_id[:8]}")

In [ ]:
# for text in left_to_langextract_inference:
#     try:
#         file_id = text_to_uuid(text).hex
#         file_path_gpt = output_dir_gpt / f"{file_id}.jsonl"
#         file_path_qwen = output_dir_qwen / f"{file_id}.jsonl"

#         result_gpt = lx.extract(
#             text_or_documents=text,
#             prompt_description=prompt,
#             examples=examples,
#             model=model_gpt,
#             use_schema_constraints=False
#         )
        
#         output_data = {
#             "extractions": [
#                 {
#                     "extraction_class": e.extraction_class,
#                     "extraction_text": e.extraction_text,
#                     "char_interval": {
#                         "start_pos": e.char_interval.start_pos if e.char_interval else None,
#                         "end_pos": e.char_interval.end_pos if e.char_interval else None
#                     },
#                     "alignment_status": e.alignment_status.value if hasattr(e, 'alignment_status') else None,
#                     "extraction_index": e.extraction_index,
#                     "group_index": e.group_index,
#                     "description": e.description,
#                     "attributes": e.attributes if hasattr(e, 'attributes') else {}
#                 } for e in result_gpt.extractions
#             ],
#             "text": text,
#             "document_id": file_id
#         }
        
#         with open(file_path_gpt, "w", encoding="utf-8") as f:
#             f.write(json.dumps(output_data, ensure_ascii=False) + "\n")
            
#         print(f"Saved in: {file_path_gpt}")

#     except Exception as e:
#         print(f"Error processing text ({text[:30]}...): {e} with GPT")

#         result_qwen = lx.extract(
#             text_or_documents=text,
#             prompt_description=prompt,
#             examples=examples,
#             model=model_qwen,
#             use_schema_constraints=False
#         )
        
#         output_data = {
#             "extractions": [
#                 {
#                     "extraction_class": e.extraction_class,
#                     "extraction_text": e.extraction_text,
#                     "char_interval": {
#                         "start_pos": e.char_interval.start_pos if e.char_interval else None,
#                         "end_pos": e.char_interval.end_pos if e.char_interval else None
#                     },
#                     "alignment_status": e.alignment_status.value if hasattr(e, 'alignment_status') else None,
#                     "extraction_index": e.extraction_index,
#                     "group_index": e.group_index,
#                     "description": e.description,
#                     "attributes": e.attributes if hasattr(e, 'attributes') else {}
#                 } for e in result_qwen.extractions
#             ],
#             "text": text,
#             "document_id": file_id
#         }
        
#         with open(file_path_qwen, "w", encoding="utf-8") as f:
#             f.write(json.dumps(output_data, ensure_ascii=False) + "\n")
            
#         print(f"Saved in: {file_path_qwen}")


#     except Exception as e:
#         print(f"Error processing text ({text[:30]}...): {e} with Qwen")

In [ ]:
# test_output_dir = Path("langextract-outputs")
# test_outputs = [file for file in list(test_output_dir.iterdir()) if file.suffix == ".jsonl"]
# len(test_outputs)

In [ ]:
# # Display a random sample of the extracted data
# random_sample = np.random.choice(test_outputs)
# document_id = random_sample.stem

# extracted_data = next(
#     lx.io.load_annotated_documents_jsonl(output_dir / f"{document_id}.jsonl"), None
# )
# pprint(extracted_data)

In [ ]:
with open(f"ner-trainset-evaluation/langextract/predictions.jsonl", "r") as f:
    lx_predictions = [json.loads(line) for line in f]

In [ ]:
# lx_predictions[0].get("sample_id", [])

In [ ]:
# lx_predictions[0].get("raw_payload", []).get("text", "")

In [ ]:
def build_token_offsets(text, tokens):
    offsets = []
    cursor = 0
    for token in tokens:
        start = text.find(token, cursor)
        if start == -1:
            raise ValueError(
                f"No pude alinear token '{token}' en '{text[cursor : cursor + 50]}'"
            )
        end = start + len(token)
        offsets.append((start, end))
        cursor = end
    return offsets

In [ ]:
# text = extracted_data.text

# token_offsets = build_token_offsets(text, text.split())
# for token, (start, end) in zip(text.split(), token_offsets):
#     assert text[start:end] == token, f"Token mismatch: '{text[start:end]}' != '{token}'"
#     print(f"'{token}': ({start}, {end})")

In [ ]:
def extractions_to_bio(extractions, token_offsets, default_label="O", mode=None):
    labels = [default_label] * len(token_offsets)
    for extraction in extractions:
        # Skip extractions without character intervals
        if mode == 'langextract':
            if not extraction.get("char_interval"):
                continue
            cls = extraction.get("extraction_class")
            start_char = extraction.get("char_interval", {}).get("start_pos")
            end_char = extraction.get("char_interval", {}).get("end_pos")
            first = True
            for i, (tok_start, tok_end) in enumerate(token_offsets):
                if end_char <= tok_start or start_char >= tok_end:
                    continue
                prefix = "B-" if first else "I-"
                labels[i] = f"{prefix}{cls}"
                first = False
        elif mode == 'ner-api':
            cls = extraction.get("attrs").get("aymurai_label")
            start_char = extraction.get("start_char")
            end_char = extraction.get("end_char")
            first = True
            for i, (tok_start, tok_end) in enumerate(token_offsets):
                if end_char <= tok_start or start_char >= tok_end:
                    continue
                prefix = "B-" if first else "I-"
                labels[i] = f"{prefix}{cls}"
                first = False
    return labels

In [ ]:
# pprint(extractions_to_bio(extracted_data.extractions, token_offsets))

In [ ]:
test_map = {
    text_to_uuid(text).hex: {"text": text, "labels": labels} for text, labels in test
}

train_map = {
    text_to_uuid(text).hex: {"text": text, "labels": labels} for text, labels in train
}

In [ ]:
train_map

In [ ]:
# with open(f"ner-evaluations/test-evaluation.txt", "r") as f:
#     ner_test_text = f.readlines()

In [ ]:
# def parse_eval_paragraphs(lines):
#     paragraphs = []
#     current_tokens = []
#     current_labels = []
#     current_evals = []

#     for raw_line in lines:
#         line = raw_line.rstrip()
        
#         if not line:
#             if current_tokens:
#                 paragraph_text = " ".join(current_tokens).strip()
#                 paragraphs.append((paragraph_text, list(current_labels), list(current_evals)))
#                 current_tokens = []
#                 current_labels = []
#                 current_evals = []
#             continue

#         parts = line.rsplit(" ", 2)
        
#         if len(parts) == 3:
#             token, label, evaluation = parts
#         elif len(parts) == 2:
#             token, label = parts
#             evaluation = ""
#         else:
#             token = line
#             label = ""
#             evaluation = ""

#         current_tokens.append(token)
#         current_labels.append(label.strip())
#         current_evals.append(evaluation.strip())
        
#     if current_tokens:
#         paragraph_text = " ".join(current_tokens).strip()
#         paragraphs.append((paragraph_text, list(current_labels), list(current_evals)))

#     return paragraphs

In [ ]:
# ner_test = parse_eval_paragraphs(ner_test_text)
# len(ner_test)

In [ ]:
# ner_test[30]

In [ ]:
def evaluate_sample(sample_text, gold_labels, annotated_document):
    tokens = sample_text.split()
    token_offsets = build_token_offsets(sample_text, tokens)
    pred_labels = extractions_to_bio(annotated_document.extractions, token_offsets)
    return gold_labels, pred_labels

In [ ]:
# text = test_map[document_id]["text"]
# labels = test_map[document_id]["labels"]
# evaluate_sample(text, labels, extracted_data)

In [ ]:
def load_prediction(doc_id):
    output_dir_4 = Path("langextract-outputs-full")
    path = output_dir_4 / f"{doc_id}.jsonl"
    if not path.exists():
        return None
    return next(lx.io.load_annotated_documents_jsonl(path), None)


# test_predictions = {doc_id: load_prediction(doc_id) for doc_id in test_map.keys()}
# missing_predictions = [doc_id for doc_id, pred in test_predictions.items() if pred is None]

In [ ]:
def generate_bio_corpus(data_dict, output_path, tokenizer_fn=None, mode=None):
    """
    Processes a dictionary of texts and extractions to create a BIO-tagged file.
    
    Args:
        data_dict (dict): { "original text": [extraction_obj1, ...], ... }
        output_path (str): Path to the resulting .txt file.
        tokenizer_fn (callable, optional): A function that takes a string and returns 
                                           a list of tokens. Defaults to str.split().
        mode (str, optional): The mode for generating the BIO corpus. Defaults to None.
    """
    corpus_lines = []

    for text, extractions in data_dict.items():
        tokens = tokenizer_fn(text) if tokenizer_fn else text.split()

        try:
            offsets = build_token_offsets(text, tokens)
        except ValueError as e:
            print(f"Skipping text due to alignment error: {e}")
            continue

        bio_labels = extractions_to_bio(extractions, offsets, mode=mode)

        for token, label in zip(tokens, bio_labels):
            corpus_lines.append(f"{token} {label}")
        
        corpus_lines.append("")

    # Save the final corpus
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(corpus_lines))
    
    print(f"BIO Corpus successfully generated at: {output_path}")

In [ ]:
lx_train_preds = {
    pred.get("raw_payload", {}).get("text", ""): pred.get("raw_payload", {}).get("extractions", "")
    for pred in lx_predictions
}

In [ ]:
with open(f"ner-trainset-evaluation/ner-api/predictions.jsonl", "r") as f:
    ner_predictions = [json.loads(line) for line in f]

In [ ]:
ner_train_preds = {
    pred.get("raw_payload", {}).get("document", ""): pred.get("raw_payload", {}).get("labels", "")
    for pred in ner_predictions
}

In [ ]:
generate_bio_corpus(data_dict=lx_train_preds, output_path="langextract-evaluations/train-evaluation-beta.txt", mode='langextract')

In [ ]:
generate_bio_corpus(ner_train_preds, output_path="ner-evaluations/train-evaluation-new.txt", mode='ner-api')

In [ ]:
output_dir_5 = Path("langextract-evaluations")
output_dir_5.mkdir(parents=True, exist_ok=True)
file_path = output_dir_5 / "test-evaluation.txt"

In [ ]:
df_2 = pd.read_csv(file_path, sep="\s", header=None)
df_2.columns = ["token", "pred"]
df_2.head(15)

In [ ]:
lines_with_errors = []

with open("langextract-evaluations/train-evaluation.txt", "r", encoding="utf-8") as f:
    for num_linea, line in enumerate(f, 1):
        line_stripped = line.strip()
        
        if not line_stripped:
            continue
            
        columnas = line_stripped.split()
        
        if len(columnas) != 2:
            lines_with_errors.append({
                "linea": num_linea,
                "columnas_detectadas": len(columnas),
                "contenido_glorificado": repr(line)  # repr muestra espacios/tabulaciones explícitos
            })

print(f"Se encontraron {len(lines_with_errors)} líneas con problemas estructurales.\n")
for error in lines_with_errors:
    print(f"Fila {error['linea']} (Columnas vistas: {error['columnas_detectadas']}): {error['contenido_glorificado']}")

In [ ]:
# read .txt file and split into tokens and tags, keeping empty lines as paragraph separators
def load_bio_txt(file_path: str, num_labels: int = 1) -> pd.DataFrame:
    """
    Reads a BIO/CoNLL formatted text file and converts it into a pandas DataFrame,
    preserving empty lines (paragraph separators) as rows with None values.
    Supports a variable number of label columns.
    
    Parameters:
    file_path (str): Path to the .txt file.
    num_labels (int): Number of label columns in the file (default is 1).
    
    Returns:
    pd.DataFrame: A DataFrame with 'Token' and 'Tag_1', 'Tag_2', etc. columns.
    """
    lines = []
    expected_cols = 1 + num_labels
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                lines.append([None] * expected_cols)
            else:
                parts = line.strip().split()
                if len(parts) == expected_cols:
                    lines.append(parts)
                else:
                    continue
                    
    col_names = ["Token"] + [f"Tag_{i+1}" for i in range(num_labels)]
    
    return pd.DataFrame(lines, columns=col_names)

# df_train_lx = load_bio_txt("langextract-evaluations/train-evaluation.txt", num_labels=1)
# df_train_ner = load_bio_txt("ner-evaluations/train-evaluation.txt", num_labels=2)

In [ ]:
df_train_lx.columns = ["token_lx", "lx_pred"]
df_train_lx.head(15)

In [ ]:
df_train_ner.columns = ["token_ner", "label", "ner_pred"]
df_train_ner.head(15)

In [ ]:
joined_df_train = df_train_ner.join(df_train_lx, lsuffix='_ner', rsuffix='_lx')

mismatches = joined_df_train[(joined_df_train['token_ner'] != joined_df_train['token_lx']) & (joined_df_train['token_ner'].notna()) & (joined_df_train['token_lx'].notna())]

if not mismatches.empty:
    print(f"Warning: Found {len(mismatches)} token mismatches!")
    print(mismatches.head())
else:
    print("Perfect alignment: All tokens match by index.")

    joined_df_train = df_train_ner.join(df_train_lx, lsuffix='_ner', rsuffix='_lx')

print(joined_df_train.head())

In [ ]:
# df_test.head(15)

In [ ]:
# len(df_test)

In [ ]:
# len(df_2)

In [ ]:
# df_1 = df_test.rename(columns={df_test.columns[0]: 'token_1'})
# df_2_renamed = df_2.rename(columns={df_2.columns[0]: 'token_2'})

# joined_df = df_1.join(df_2_renamed, lsuffix='_original', rsuffix='_langextract')

# mismatches = joined_df[joined_df['token_1'] != joined_df['token_2']]

# if not mismatches.empty:
#     print(f"Warning: Found {len(mismatches)} token mismatches!")
#     print(mismatches.head())
# else:
#     print("Perfect alignment: All tokens match by index.")

#     joined_df = df_test.join(df_2, lsuffix='_original', rsuffix='_langextract')

# print(joined_df.head())

In [ ]:
joined_df_train = joined_df_train.drop(columns=['token_lx'])

In [ ]:
joined_df_train.head()

In [ ]:
joined_df_train['match'] = (
    (joined_df_train['label'] == joined_df_train['ner_pred']) & 
    (joined_df_train['ner_pred'] == joined_df_train['lx_pred'])
)

In [ ]:
# Apply only where the value is not null
joined_df_train["normalized_label"] = joined_df_train["label"].apply(lambda x: normalize_class(x) if pd.notna(x) else x)
joined_df_train["normalized_ner"] = joined_df_train["ner_pred"].apply(lambda x: normalize_class(x) if pd.notna(x) else x)
joined_df_train["normalized_lx_pred"] = joined_df_train["lx_pred"].apply(lambda x: normalize_class(x) if pd.notna(x) else x)

joined_df_train.head(30)

In [ ]:
joined_df_train = joined_df_train.drop(columns=['label','ner_pred', 'lx_pred'])

In [ ]:
joined_df_train.rename(columns={
    "normalized_label": "label",
    "normalized_ner": "ner_pred",
    "normalized_lx_pred": "lx_pred"
}, inplace=True)

In [ ]:
joined_df_train

### PER label analysis

In [ ]:
# cases where lx_pred is PER but the token is not labeled as such, and also does not start with a capital letter or matches common false positives for PER entities in legal texts.

regex_exclude_per = r'^(Fiscalía|Licenciado|DESCONOCIDA|Lado|RE|ZORRA|T.?|N[.]N[.]?|Defensora|La|Testigo|Cámara|Unidad|Sur|Sr|Sra|Srta|Dr|Lic|F°63|Dra|Inspector|OTROS|Y|Oficial|Defensor|–Oficial|Presunta|Fiscal|Comisario|Cabo|Ministerio|Público|Beat|NN|Patronato|Liberados|Primero|Sargento|Mayor|Principal|Auxiliar|Juez)\b'

joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'PER') &
    (joined_df_train['ner_pred'].isin(['PER', 'O'])) &
    (joined_df_train['token_ner'].str.contains(r'^[A-ZÑÁÉÍÓÚ]', na=False)) &
    (~joined_df_train['token_ner'].str.contains(regex_exclude_per, na=False))
    ].index

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_12052613.parquet")

In [ ]:
df_reviewed[df_reviewed.label == 'NUM_EXPEDIENTE'][:50]

In [ ]:
445869

In [ ]:
df_reviewed[
    (df_reviewed.label != df_reviewed.lx_pred) & 
    (~df_reviewed.label.isin(['O','TEXTO_ANONIMIZAR'])) &
    (~df_reviewed.lx_pred.isin(['O'])) &
    (df_reviewed.label.notna())
][:50]

In [ ]:
df_reviewed = joined_df_train.copy()

mask = (
    (df_reviewed['label'] == 'O') &
    (df_reviewed['lx_pred'] == 'PER') &
    (df_reviewed['ner_pred'].isin(['PER', 'O'])) &
    (df_reviewed['token_ner'].str.contains(r'^[A-ZÑÁÉÍÓÚ]', na=False)) &
    (~df_reviewed['token_ner'].str.contains(regex_exclude_per, na=False))
)

per_indices = df_reviewed[mask].index

df_reviewed.loc[mask, 'label'] = df_reviewed.loc[mask, 'lx_pred']

print(f"Rows updated: {mask.sum()}")

In [ ]:
df_reviewed[(df_reviewed.index.isin(per_indices))]

### FECHA label analysis

In [ ]:
regex_exclude_date = r'^(fines|del|otras|audiencias|que|debe|participar|asado|una|quema|estupefacientes|a|años|lunes|martes|miércoles|jueves|viernes|sábado|domingo|el|la|descripción|física|día|días|mañana|desde|hasta|determinada|no|aún|hoy|un|fin|semana|largo|aproximadamente|am|pm|las|ayer|siendo|hoy|entre|los|fecha|hora|horas|\d{1,2}:\d{2}(\s*hs)?|principios|mediados|local|denuncia)\b'
# Regex to match single-line date formats like 5/4/26, 05/04/2026, 5-4-26, etc.
regex_single_line_date = r'^\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}[.,;]?\s*$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'FECHA') &
    (~joined_df_train['token_ner'].str.contains(regex_exclude_date, case=False, na=False))
].copy()

indices = filtered_df.index.to_series()

diff_back_1 = indices.diff(1)
diff_back_2 = indices.diff(2)

diff_fwd_1 = indices.diff(-1).abs()
diff_fwd_2 = indices.diff(-2).abs()

is_start_of_three = (diff_fwd_1 == 1) & (diff_fwd_2 == 2)

is_middle_of_three = (diff_back_1 == 1) & (diff_fwd_1 == 1)

is_end_of_three = (diff_back_1 == 1) & (diff_back_2 == 2)

is_streak_of_three = is_start_of_three | is_middle_of_three | is_end_of_three

is_full_date = filtered_df['token_ner'].str.contains(regex_single_line_date, na=False)

final_condition = is_streak_of_three | is_full_date

date_indices = filtered_df[final_condition].index

filtered_df[final_condition][1500:1600]

In [ ]:
df_reviewed.loc[date_indices, 'label'] = df_reviewed.loc[date_indices, 'lx_pred']

In [ ]:
df_reviewed[(df_reviewed.index.isin(date_indices))][:100]

### LOC label analysis

In [ ]:
regex_exclude_loc = r'^(GCABA|GCBA|G\.C\.B\.A\b|G.C.A.B.A|Gobierno|GCBA)\b'

allowed_lowercase = r'^(de|del|la|las|el|los|en|y)$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'LOC') &
    (~joined_df_train['token_ner'].str.contains(regex_exclude_loc, case=False, na=False))
].copy()

indices = filtered_df.index.to_series()
is_new_group = indices.diff() != 1
filtered_df['entity_group_id'] = is_new_group.cumsum()


is_lowercase = filtered_df['token_ner'].str.contains(r'^[a-zñáéíóú]+$', na=False)

is_allowed_connector = filtered_df['token_ner'].str.contains(allowed_lowercase, case=False, na=False)

filtered_df['is_illegal_lowercase'] = is_lowercase & ~is_allowed_connector

group_has_illegal_word = filtered_df.groupby('entity_group_id')['is_illegal_lowercase'].transform('any')

final_condition = ~group_has_illegal_word

filtered_df[['token_ner','label','ner_pred','lx_pred']][final_condition][1519:1560]

In [ ]:
loc_indices = filtered_df[['token_ner','label','ner_pred','lx_pred']][final_condition][:13].index

In [ ]:
loc_indices

In [ ]:
loc_indices = loc_indices.append(filtered_df[['token_ner','label','ner_pred','lx_pred']][final_condition][1519:1560].index)

loc_indices

In [ ]:
df_reviewed.loc[loc_indices, 'label'] = df_reviewed.loc[loc_indices, 'lx_pred']

### Checkpoint saving LOC

In [ ]:
# output_dir = "reviewing-data"
# os.makedirs(output_dir, exist_ok=True)

# df_path = os.path.join(output_dir, "df_reviewed_070526.parquet")
# indices_path = os.path.join(output_dir, "loc_indices_070526.joblib")

# df_reviewed.to_parquet(df_path, index=True)


# joblib.dump(loc_indices, indices_path)

# print(f"Successfully saved to {output_dir}/")
# print(f"Saved {len(loc_indices)} indices for locations.")

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_070526.parquet")
loc_indices = joblib.load("reviewing-data/loc_indices_070526.joblib")

In [ ]:
df_reviewed[(df_reviewed.index.isin(loc_indices))][:100]

### NUM_EXPEDIENTE label analysis

In [ ]:
joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NUM_EXPEDIENTE')
    ]

In [ ]:
regex_include_exp = r'^\d+/\d{2,4}(-\d+)?|IPP'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O')
    & (joined_df_train['lx_pred'] == 'O')
    & (joined_df_train['token_ner'].str.contains(regex_include_exp, case=False, na=False))
].copy()

filtered_df[95:]

In [ ]:
exp_indices = filtered_df.index

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NUM_EXPEDIENTE') &
    (~joined_df_train['token_ner'].str.contains(regex_include_exp, case=False, na=False))
].copy()

filtered_df

In [ ]:
date_indices = date_indices.append(filtered_df[93:94].index)

In [ ]:
exp_indices = exp_indices.append(filtered_df[95:].index)

exp_indices

In [ ]:
df_reviewed.loc[exp_indices, 'label'] = 'NUM_EXPEDIENTE'

In [ ]:
df_reviewed.loc[date_indices, 'label'] = 'FECHA'

In [ ]:
df_path = os.path.join("reviewing-data", "df_reviewed_080526.parquet")
indices_path = os.path.join("reviewing-data", "exp_indices_080526.joblib")

df_reviewed.to_parquet(df_path, index=True)


joblib.dump(exp_indices, indices_path)

print("Successfully saved to reviewing-data/")
print(f"Saved {len(exp_indices)} indices for archival.")

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_080526.parquet")

In [ ]:
exp_indices = joblib.load("reviewing-data/exp_indices_080526.joblib")

### CUIJ label analysis

In [ ]:
joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'CUIJ')
    ]

### NUM_ACTUACION label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O')
    # & (joined_df_train['lx_pred'] == 'NUM_ACTUACION')
    & (joined_df_train['token_ner'].str.contains(regex_include_exp, case=False, na=False))
    ].copy()

filtered_df[:30]

In [ ]:
exp_indices = exp_indices.append(filtered_df.index)

In [ ]:
df_reviewed.loc[exp_indices, 'label'] = df_reviewed.loc[exp_indices, 'lx_pred']

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NUM_ACTUACION')
    & (~joined_df_train['token_ner'].str.contains(regex_include_exp, case=False, na=False))
    ].copy()

filtered_df[56:70]

In [ ]:
act_indices = filtered_df[15:16].index

In [ ]:
act_indices = act_indices.append(filtered_df[54:56].index)

In [ ]:
df_reviewed.loc[act_indices, 'label'] = df_reviewed.loc[act_indices, 'lx_pred']

In [ ]:
df_path = os.path.join("reviewing-data", "df_reviewed_09052601.parquet")
indices_path = os.path.join("reviewing-data", "act_indices_09052601.joblib")

df_reviewed.to_parquet(df_path, index=True)


joblib.dump(act_indices, indices_path)

print("Successfully saved to reviewing-data/")
print(f"Saved {len(act_indices)} indices for archival.")

### DNI label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'DNI')
    ].copy()

filtered_df

### LINK label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'LINK')
    ].copy()

filtered_df[13:15]

In [ ]:
link_indices = filtered_df[11:12].index

In [ ]:
link_indices = link_indices.append(filtered_df[13:15].index)

In [ ]:
df_reviewed.loc[link_indices, 'label'] = df_reviewed.loc[link_indices, 'lx_pred']

In [ ]:
df_path = os.path.join("reviewing-data", "df_reviewed_09052602.parquet")
indices_path = os.path.join("reviewing-data", "link_indices_09052602.joblib")

df_reviewed.to_parquet(df_path, index=True)


joblib.dump(link_indices, indices_path)

print("Successfully saved to reviewing-data/")
print(f"Saved {len(link_indices)} indices for archival.")

### DIRECCION label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'DIRECCION') &
     (joined_df_train['ner_pred'] == 'DIRECCION')
    ].copy()

filtered_df

In [ ]:
dir_indices = filtered_df.index

In [ ]:
regex_excluded_words_dir = r'^(Buenos|Aires|Provincia|Ciudad|CABA|PBA)\W*$'

excluded_words_group = [
    'esta', 'del', 'calle', 'de', 'la', 'intersección', 'avenida', 
    'esquina', 'entre', 'y', 'número', 'nro', 'n°', 'manzana', 
    'lote', 'parcela', 'barrio', 'localidad', 'país', 'ciudad'
]
regex_excluded_words_groups = r'^(' + '|'.join(excluded_words_group) + r')\W*$'

df_step1 = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'DIRECCION') &
    (joined_df_train['ner_pred'] != 'DIRECCION') &
    (~joined_df_train['token_ner'].str.contains(regex_excluded_words_dir, case=False, na=False))
].copy()

indices = df_step1.index.to_series()
is_new_group = indices.diff() != 1
df_step1['entity_group_id'] = is_new_group.cumsum()

df_step1['is_excluded_token'] = df_step1['token_ner'].str.contains(
    regex_excluded_words_groups, case=False, na=False
)

group_all_excluded = df_step1.groupby('entity_group_id')['is_excluded_token'].transform('all')

final_df = df_step1[~group_all_excluded].copy()

final_df

In [ ]:
# dir_indices = joblib.load("reviewing-data/dir_indices_090526.joblib")
# link_indices = joblib.load("reviewing-data/link_indices_09052602.joblib")

In [ ]:
dir_indices = dir_indices.append(final_df[649:655].index)

In [ ]:
dir_indices

In [ ]:
# indices_path = os.path.join("reviewing-data", "dir_indices_110526.joblib")
# joblib.dump(dir_indices, indices_path)
# print(f"Saved {len(dir_indices)} indices for archival.")

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_09052602.parquet")

In [ ]:
df_reviewed.loc[dir_indices, 'label'] = df_reviewed.loc[dir_indices, 'lx_pred']

In [ ]:
df_path = os.path.join("reviewing-data", "df_reviewed_11052601.parquet")

df_reviewed.to_parquet(df_path, index=True)

### We have to see the cases where langextract decided the tokens as DIR and they may be LOC instead.

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'DIRECCION') &
     (joined_df_train['ner_pred'] == 'LOC')
    ].copy()

filtered_df

In [ ]:
loc_indices = loc_indices.append(filtered_df.index)

In [ ]:
loc_indices = joblib.load("reviewing-data/loc_indices_070526.joblib")

In [ ]:
regex_excluded_words_dir = r'^(calle|esta)\W*$'

filtered_df = df_reviewed[
    (df_reviewed['label'] == 'O') &
    (df_reviewed['lx_pred'] == 'DIRECCION') &
    (~df_reviewed['token_ner'].str.contains(regex_excluded_words_dir, case=False, na=False))
    ].copy()

filtered_df[687:]

In [ ]:
loc_indices = loc_indices.append(filtered_df[687:].index)

In [ ]:
loc_indices

In [ ]:
indices_path = os.path.join("reviewing-data", "loc_indices_11052601.joblib")
joblib.dump(loc_indices, indices_path)
print(f"Saved {len(loc_indices)} indices for archival.")

In [ ]:
df_reviewed.loc[loc_indices, 'label'] = 'LOC'
df_reviewed.loc[index_to_clean, 'label'] = 'O'

In [ ]:
df_path = os.path.join("reviewing-data", "df_reviewed_11052602.parquet")

df_reviewed.to_parquet(df_path, index=True)

In [ ]:
df_filtered = df_reviewed.loc[loc_indices][df_reviewed.lx_pred == 'DIRECCION']

df_filtered[30:50]


In [ ]:
index_to_clean = df_filtered[:11].index

In [ ]:
index_to_clean = index_to_clean.append(df_filtered[691:701].index)

In [ ]:
loc_indices = loc_indices.difference(index_to_clean)

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_11052602.parquet")
index_to_clean = joblib.load("reviewing-data/index_to_clean_110526.joblib")
loc_indices = joblib.load("reviewing-data/loc_indices_11052601.joblib")

In [ ]:
indices_path = os.path.join("reviewing-data", "index_to_clean_120526.joblib")
joblib.dump(index_to_clean, indices_path)
print(f"Saved {len(index_to_clean)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052601.parquet")

df_reviewed.to_parquet(df_path, index=True)

### TELEFONO label analysis

In [ ]:
regex_tel = r'(\+?\d{1,3}[-.\s]?)?\(?\d{2,5}\)?[-.\s]?\d{3,4}[-.\s]?\d{4}|911'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'TELEFONO')
    & (joined_df_train['lx_pred'] != 'TELEFONO')
    # & (~joined_df_train['token_ner'].str.contains(regex_tel, na=False))
].copy()

filtered_df

In [ ]:
tel_indices = filtered_df.index

In [ ]:
tel_indices = tel_indices.append(filtered_df[3:].index)

In [ ]:
tel_indices

In [ ]:
df_reviewed.loc[tel_indices, 'label'] = 'TELEFONO'

In [ ]:
df_reviewed.loc[tel_indices]

In [ ]:
indices_path = os.path.join("reviewing-data", "tel_indices_120526.joblib")
joblib.dump(tel_indices, indices_path)
print(f"Saved {len(tel_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052602.parquet")

df_reviewed.to_parquet(df_path, index=True)

### CORREO_ELECTRONICO label analysis

In [ ]:
regex_mail = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,6}'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'CORREO_ELECTRONICO')
    # & (joined_df_train['token_ner'].str.contains(regex_mail, na=False, case=False))
].copy()

filtered_df

In [ ]:
mail_indices = filtered_df.index

In [ ]:
mail_indices = mail_indices.append(filtered_df.index)

In [ ]:
mail_indices

In [ ]:
df_reviewed.loc[mail_indices, 'label'] = 'CORREO_ELECTRONICO'

In [ ]:
indices_path = os.path.join("reviewing-data", "mail_indices_120526.joblib")
joblib.dump(mail_indices, indices_path)
print(f"Saved {len(mail_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052603.parquet")

df_reviewed.to_parquet(df_path, index=True)

### USUARIX label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'USUARIX')
    ].copy()

filtered_df

### NOMBRE_ARCHIVO label analysis

In [ ]:
regex_arch = r'^\w+.*\.(jpg|jpeg|img|pdf|docx?|xlsx?|pptx?|txt|csv|zip|rar|7z|tar\.gz|tar\.bz2|tar\.xz)\W*$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NOMBRE_ARCHIVO') &
    (~joined_df_train['token_ner'].str.contains(regex_arch, na=False, case=False))
].copy()

filtered_df

In [ ]:
arch_indices = filtered_df.index

In [ ]:
df_reviewed.loc[arch_indices, 'label'] = 'NOMBRE_ARCHIVO'

In [ ]:
indices_path = os.path.join("reviewing-data", "arch_indices_120526.joblib")
joblib.dump(arch_indices, indices_path)
print(f"Saved {len(arch_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052604.parquet")

df_reviewed.to_parquet(df_path, index=True)

In [ ]:
regex_fecha_iso = r'^\d{4}[- /]\d{2}[- /]\d{2}\W*$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NOMBRE_ARCHIVO') &
    (joined_df_train['token_ner'].str.contains(regex_fecha_iso, na=False, case=False))
].copy()

filtered_df

In [ ]:
date_indices = filtered_df.index

In [ ]:
df_reviewed.loc[date_indices, 'label'] = 'FECHA'

In [ ]:
indices_path = os.path.join("reviewing-data", "date_indices_120526.joblib")
joblib.dump(date_indices, indices_path)
print(f"Saved {len(date_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052605.parquet")

df_reviewed.to_parquet(df_path, index=True)

### NACIONALIDAD label analysis

In [ ]:
regex_no_caps = r'^[^A-ZÑ]*$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'NACIONALIDAD')
    # & (joined_df_train['token_ner'].str.contains(regex_no_caps, na=False))
    ].copy()

filtered_df

In [ ]:
nat_indices = filtered_df.index

In [ ]:
nat_indices = nat_indices.append(filtered_df[1:].index)

In [ ]:
df_reviewed.loc[nat_indices, 'label'] = 'NACIONALIDAD'

In [ ]:
indices_path = os.path.join("reviewing-data", "nat_indices_120526.joblib")
joblib.dump(nat_indices, indices_path)
print(f"Saved {len(nat_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052606.parquet")

df_reviewed.to_parquet(df_path, index=True)

### EDAD label analysis

In [ ]:
excluded_edad_words = ['edad', 'años', 'de', 'aproximadamente']
regex_exclude_edad = r'^(' + '|'.join(excluded_edad_words) + r')\W*$'

filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'EDAD') &
    (joined_df_train['ner_pred'] != 'EDAD') &
    (~joined_df_train['token_ner'].str.contains(regex_exclude_edad, case=False, na=False))
].copy()


filtered_df[27:28]

In [ ]:
joined_df_train[463596:463600]

In [ ]:
age_indices = filtered_df[:3].index

In [ ]:
age_indices = age_indices.append(filtered_df[27:28].index)

In [ ]:
age_indices

In [ ]:
df_reviewed.loc[age_indices, 'label'] = 'EDAD'

In [ ]:
indices_path = os.path.join("reviewing-data", "age_indices_120526.joblib")
joblib.dump(age_indices, indices_path)
print(f"Saved {len(age_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052607.parquet")

df_reviewed.to_parquet(df_path, index=True)

### BANCO label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'BANCO')
    ].copy()

filtered_df

In [ ]:
bank_indices = filtered_df[1:2].index

In [ ]:
bank_indices = bank_indices.append(filtered_df[45:59].index)

In [ ]:
bank_indices

In [ ]:
df_reviewed.loc[bank_indices, 'label'] = 'BANCO'

In [ ]:
indices_path = os.path.join("reviewing-data", "bank_indices_120526.joblib")
joblib.dump(bank_indices, indices_path)
print(f"Saved {len(bank_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052608.parquet")

df_reviewed.to_parquet(df_path, index=True)

### NUM_CAJA_AHORRO label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NUM_CAJA_AHORRO')
    ].copy()

filtered_df[:]

In [ ]:
account_indices = filtered_df[:2].index

In [ ]:
df_reviewed.loc[account_indices, 'label'] = df_reviewed.loc[account_indices, 'lx_pred']

In [ ]:
indices_path = os.path.join("reviewing-data", "account_indices_120526.joblib")
joblib.dump(account_indices, indices_path)
print(f"Saved {len(account_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052609.parquet")

df_reviewed.to_parquet(df_path, index=True)

### NUM_MATRICULA label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'NUM_MATRICULA')
    ].copy()

filtered_df[31:]

In [ ]:
mat_indices = filtered_df[2:11].index

In [ ]:
mat_indices = mat_indices.append(filtered_df[2540:2544].index)

In [ ]:
df_reviewed.loc[mat_indices, 'label'] = 'NUM_MATRICULA'

In [ ]:
indices_path = os.path.join("reviewing-data", "mat_indices_120526.joblib")
joblib.dump(mat_indices, indices_path)
print(f"Saved {len(mat_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052610.parquet")

df_reviewed.to_parquet(df_path, index=True)

### ESTUDIOS label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['ner_pred'] == 'ESTUDIOS')
    ].copy()

filtered_df[2:4]

In [ ]:
study_indices = filtered_df[:15].index

In [ ]:
study_indices = study_indices.append(filtered_df[2:4].index)

In [ ]:
study_indices

In [ ]:
df_reviewed.loc[study_indices, 'label'] = 'ESTUDIOS'

In [ ]:
indices_path = os.path.join("reviewing-data", "study_indices_120526.joblib")
joblib.dump(study_indices, indices_path)
print(f"Saved {len(study_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052611.parquet")

df_reviewed.to_parquet(df_path, index=True)

### MARCA_AUTOMOVIL label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O')
    & (joined_df_train['lx_pred'] == 'MARCA_AUTOMOVIL')
    ].copy()

filtered_df[33:]

In [ ]:
aut_indices = filtered_df[:1].index

In [ ]:
aut_indices = aut_indices.append(filtered_df[31:33].index)

In [ ]:
df_reviewed.loc[aut_indices, 'label'] = 'MARCA_AUTOMOVIL'

In [ ]:
indices_path = os.path.join("reviewing-data", "aut_indices_120526.joblib")
joblib.dump(aut_indices, indices_path)
print(f"Saved {len(aut_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052612.parquet")

df_reviewed.to_parquet(df_path, index=True)

### TEXTO_ANONIMIZAR label analysis

In [ ]:
filtered_df = joined_df_train[
    (joined_df_train['label'] == 'O') &
    (joined_df_train['lx_pred'] == 'TEXTO_ANONIMIZAR')
    ].copy()

filtered_df[3500:]

In [ ]:
text_anon_indices = filtered_df[23:25].index

In [ ]:
text_anon_indices = text_anon_indices.append(filtered_df[767:774].index)

In [ ]:
text_anon_indices

In [ ]:
df_reviewed.loc[text_anon_indices, 'label'] = 'TEXTO_ANONIMIZAR'

In [ ]:
indices_path = os.path.join("reviewing-data", "text_anon_indices_120526.joblib")
joblib.dump(text_anon_indices, indices_path)
print(f"Saved {len(text_anon_indices)} indices for archival.")

df_path = os.path.join("reviewing-data", "df_reviewed_12052613.parquet")

df_reviewed.to_parquet(df_path, index=True)

In [ ]:
joined_df_train.lx_pred.unique()

### Now we see the changes of labels between df_reviewed and joined_df_train

In [ ]:
final_df_reviewed = df_reviewed.drop(columns=['match', 'ner_pred', 'lx_pred'])

In [ ]:
final_df_reviewed.rename(columns={'token_ner':'token'}, inplace=True)

In [ ]:
df_path = os.path.join("reviewing-data", "final_df_reviewd_120526.parquet")

final_df_reviewed.to_parquet(df_path, index=True)

In [ ]:
joined_df_train_syn = joined_df_train.drop(columns=['match', 'ner_pred', 'lx_pred'])

In [ ]:
joined_df_train_syn

In [ ]:
joined_comparison = pd.merge(
    joined_df_train_syn[['token_ner', 'label']], 
    final_df_reviewed[['token', 'label']], 
    left_index=True, 
    right_index=True, 
    how='inner', # Solo lo que está en ambos
    suffixes=('_train', '_reviewed')
)


joined_comparison['label_train'] = joined_comparison['label_train'].astype(str).replace('nan', 'O')
joined_comparison['label_reviewed'] = joined_comparison['label_reviewed'].astype(str).replace('nan', 'O')

joined_comparison['match'] = (
    joined_comparison['label_train'] == joined_comparison['label_reviewed']
)

In [ ]:
joined_comparison[joined_comparison['match'] ==  False][:10]

In [ ]:
def export_simple_df_to_bio(df, output_filename="new-data/train_review.txt"):
    bio_lines = []
    prev_label = 'O'
    
    for _, row in df.iterrows():
        token = row['token']
        label = row['label']
        
        if pd.isna(token) or token is None:
            bio_lines.append("") 
            prev_label = 'O'
            continue
            
        token_str = str(token).strip()
        
        if label == 'O' or pd.isna(label):
            bio_tag = 'O'
        else:
            if label == prev_label:
                bio_tag = f"I-{label}"
            else:
                bio_tag = f"B-{label}"
                
        bio_lines.append(f"{token_str} {bio_tag}")
        prev_label = label
        
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("\n".join(bio_lines))
        
    print(f"File '{output_filename}' created successfully with {len(bio_lines)} lines.")

In [ ]:
df_reviewed = pd.read_parquet("reviewing-data/df_reviewed_12052613.parquet")

In [ ]:
export_simple_df_to_bio(final_df_reviewed)

In [ ]:
# joined_df_clean['match'] = (
#     (joined_df_clean['label'] == joined_df_clean['pred_original']) & 
#     (joined_df_clean['pred_original'] == joined_df_clean['pred_langextract'])
#

In [ ]:
# joined_df_clean[(joined_df_clean['match'] == False) & (joined_df_clean['label'] != joined_df_clean['pred_langextract'])][1200:1250]

In [ ]:
# joined_df_clean[(joined_df_clean['match'] == False) & (joined_df_clean['label'] != joined_df_clean['pred_langextract']) & (joined_df_clean['pred_original'] != joined_df_clean['label'])][450:100]

In [ ]:
# import json
# import os

# def build_token_offsets(text, tokens):
#     offsets = []
#     cursor = 0
#     for token in tokens:
#         start = text.find(token, cursor)
#         if start == -1:
#             # Fallback for minor encoding mismatches
#             start = text.lower().find(token.lower(), cursor)
#             if start == -1:
#                 raise ValueError(f"Could not align token '{token}' in '{text[cursor : cursor + 50]}'")
#         end = start + len(token)
#         offsets.append((start, end))
#         cursor = end
#     return offsets

# def extractions_to_bio(spans, token_offsets):
#     labels = ["O"] * len(token_offsets)
#     for span in spans:
#         cls = span.get('label')
#         start_char = span.get('start')
#         end_char = span.get('end')
        
#         if start_char is None or end_char is None:
#             continue
            
#         first = True
#         for i, (tok_start, tok_end) in enumerate(token_offsets):
#             if end_char <= tok_start or start_char >= tok_end:
#                 continue
            
#             prefix = "B-" if first else "I-"
#             labels[i] = f"{prefix}{cls}"
#             first = False
#     return labels

# def generate_comparison_file(output_path):
#     gold_path = 'dev-set/langextract/artifacts/data/samples_gold.jsonl'
#     flair_path = 'dev-set/ner/artifacts/predictions/predictions.jsonl'
#     lang_path = 'dev-set/langextract/artifacts/predictions/predictions.jsonl'

#     def load_jsonl(path):
#         data = {}
#         if not os.path.exists(path): return data
#         with open(path, 'r', encoding='utf-8') as f:
#             for line in f:
#                 item = json.loads(line)
#                 data[item['sample_id']] = item
#         return data

#     gold_data = load_jsonl(gold_path)
#     flair_preds = load_jsonl(flair_path)
#     lang_preds = load_jsonl(lang_path)

#     output_lines = []
#     output_lines.append("TOKEN\tGOLD\tFLAIR\tLANGEXTRACT")
#     output_lines.append("-" * 60)

#     for sid, gold_item in gold_data.items():
#         text = gold_item.get('text', '')
#         gold_bio = gold_item.get('gold_bio', [])

#         tokens = text.split()
        
#         try:
#             offsets = build_token_offsets(text, tokens)
            
#             flair_spans = flair_preds.get(sid, {}).get('spans', [])
#             lang_spans = lang_preds.get(sid, {}).get('spans', [])
            
#             flair_bio = extractions_to_bio(flair_spans, offsets)
#             lang_bio = extractions_to_bio(lang_spans, offsets)

#             for i in range(len(tokens)):
#                 t = tokens[i]
#                 g = gold_bio[i] if i < len(gold_bio) else "O"
#                 f = flair_bio[i]
#                 l = lang_bio[i]
#                 output_lines.append(f"{t}\t{g}\t{f}\t{l}")
            
#             output_lines.append("")
            
#         except ValueError as e:
#             print(f"Skipping sample {sid}: {e}")
#             continue

#     os.makedirs(os.path.dirname(output_path), exist_ok=True)
#     with open(output_path, 'w', encoding='utf-8') as f:
#         f.write("\n".join(output_lines))
    
#     print(f"Comparison file successfully generated at: {output_path}")

# generate_comparison_file('dev-set/comparison_results.txt')

In [ ]:
# import pandas as pd

# def create_comparison_df(file_path):
#     df_dev = pd.read_csv(
#         file_path, 
#         sep='\t', 
#         skiprows=2, 
#         names=['token', 'gold', 'flair', 'langextract'],
#         skip_blank_lines=True
#     )

#     df_dev = df_dev.dropna(subset=['token'])
#     df_dev['match'] = (df_dev['gold'] == df_dev['flair']) & (df_dev['flair'] == df_dev['langextract'])
    
#     return df_dev

# df_dev = create_comparison_df('dev-set/comparison_results.txt')

# df_dev.head()

In [ ]:
df_dev[(df_dev['match'] == False) & (df_dev['gold'] != df_dev['langextract'])]

In [ ]:
print(
    f"{len(predictions) - len(missing_predictions)}/{len(predictions)} documentos con predicciones disponibles"
)

In [ ]:
!uv pip install seqeval

In [ ]:
from seqeval.metrics import classification_report

paired_sequences = {
    doc_id: evaluate_sample(
        test_map[doc_id]["text"],
        test_map[doc_id]["labels"],
        pred_doc,
    )
    for doc_id, pred_doc in predictions.items()
    if pred_doc is not None
}

paired_labels = list(paired_sequences.values())
list(paired_sequences.items())[:1]

In [ ]:
def strip_bio_sequence(seq):
    cleaned = []
    for label in seq:
        if not label or label == "O":
            cleaned.append("O")
        else:
            cleaned.append(label.split("-", 1)[-1])
    return cleaned


def compute_relaxed_metrics(gold_sequences, pred_sequences):
    gold_flat = []
    pred_flat = []
    for gold_seq, pred_seq in zip(gold_sequences, pred_sequences):
        stripped_gold = strip_bio_sequence(gold_seq)
        stripped_pred = strip_bio_sequence(pred_seq)
        gold_flat.extend(stripped_gold)
        pred_flat.extend(stripped_pred)
    labels = sorted({lab for lab in gold_flat + pred_flat if lab != "O"})
    per_label = {}
    tp_total = fp_total = fn_total = 0
    for label in labels:
        tp = sum(1 for g, p in zip(gold_flat, pred_flat) if g == label and p == label)
        fp = sum(1 for g, p in zip(gold_flat, pred_flat) if g != label and p == label)
        fn = sum(1 for g, p in zip(gold_flat, pred_flat) if g == label and p != label)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            (2 * precision * recall) / (precision + recall)
            if (precision + recall)
            else 0.0
        )
        support = sum(1 for g in gold_flat if g == label)
        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
        tp_total += tp
        fp_total += fp
        fn_total += fn
    micro_precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = (
        (2 * micro_precision * micro_recall) / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )
    accuracy = (
        sum(1 for g, p in zip(gold_flat, pred_flat) if g == p) / len(gold_flat)
        if gold_flat
        else 0.0
    )
    return (
        per_label,
        {
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": micro_f1,
        },
        accuracy,
    )

In [ ]:
if paired_labels:
    gold_all, pred_all = zip(*paired_labels)
    gold_all, pred_all = list(gold_all), list(pred_all)
else:
    gold_all, pred_all = [], []

exact_match_flags = {}

total_samples = len(paired_sequences)
print(
    f"Evaluando {total_samples} documentos (sin predicciones: {len(missing_predictions)})"
)

if gold_all:
    print("\n=== Métrica estricta (BIO) ===")
    print(classification_report(gold_all, pred_all, mode="strict"))

    relaxed_per_label, relaxed_micro, relaxed_accuracy = compute_relaxed_metrics(
        gold_all, pred_all
    )
    print("\n=== Métrica token-level sin prefijo BIO ===")
    for label in sorted(relaxed_per_label.keys()):
        stats = relaxed_per_label[label]
        print(
            f"{label:>18} | P: {stats['precision']:.3f} R: {stats['recall']:.3f} F1: {stats['f1']:.3f} (support={stats['support']})"
        )
    print(
        f"Micro -> P: {relaxed_micro['precision']:.3f} R: {relaxed_micro['recall']:.3f} F1: {relaxed_micro['f1']:.3f}"
    )
    print(f"Accuracy global: {relaxed_accuracy:.3f}")

    match_scores = {
        doc_id: (
            sum(1 for g, p in zip(gold_seq, pred_seq) if g == p) / len(gold_seq)
            if gold_seq
            else 0.0
        )
        for doc_id, (gold_seq, pred_seq) in paired_sequences.items()
    }
    print(
        f"\nPromedio de match score: {round(sum(match_score for match_score in match_scores.values()) / len(match_scores), 4)}"
    )
    exact_match_flags = {doc_id: score == 1.0 for doc_id, score in match_scores.items()}
    exact_matches = sum(1 for score in match_scores.values() if score == 1.0)
    exact_match_ratio = exact_matches / total_samples if total_samples else 0.0
    print(
        f"\nExact match por muestra: {exact_matches}/{total_samples} ({exact_match_ratio:.1%})"
    )
    preview_flags = list(match_scores.items())[:5]
    if preview_flags:
        print("Ejemplo de scores (máx. 5):", preview_flags)
    preview_flags = list(exact_match_flags.items())[:5]
    if preview_flags:
        print("Ejemplo de flags exactos (máx. 5):", preview_flags)
else:
    print("No hay pares de etiquetas para evaluar.")

if missing_predictions:
    print("Documentos sin predicción disponible (máx. 10):", missing_predictions[:10])

In [ ]:
# Filter docs with exact matches
exact_matches = [
    doc_id for doc_id, match_score in match_scores.items() if match_score == 1.0
]
inexact_matches = [
    doc_id for doc_id, match_score in match_scores.items() if match_score < 1.0
]

In [ ]:
len(exact_matches), len(inexact_matches)

In [ ]:
for doc_id in np.random.choice(
    exact_matches, size=min(3, len(exact_matches)), replace=False
):
    print(f"\n=== Documento con match exacto: {doc_id} ===")
    pprint(
        {
            token: label
            for token, label in zip(
                test_map[doc_id]["text"].split(),
                test_map[doc_id]["labels"],
            )
            if label != "O"
        }
    )
    print(f"Match score: {match_scores[doc_id]}")
    pprint(predictions[doc_id])

In [ ]:
for doc_id in np.random.choice(
    inexact_matches, size=min(3, len(inexact_matches)), replace=False
):
    print(f"\n=== Documento con match inexacto: {doc_id} ===")
    pprint(
        [
            {token: label}
            for token, label in zip(
                test_map[doc_id]["text"].split(),
                test_map[doc_id]["labels"],
            )
            if label != "O"
        ]
    )
    print(f"Match score: {match_scores[doc_id]}")
    pprint(predictions[doc_id])